# AAPL — Phase 1 Data Exploration
Run the pipeline scripts first:
```
python pipeline/01_fetch_data.py
python pipeline/02_features.py
python pipeline/03_labels.py
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

df = pd.read_parquet(Path('../data/processed/aapl_labeled.parquet'))
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')
df.tail(3)

In [ ]:
# --- Price history ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(df.index, df['close'], linewidth=0.8)
axes[0].set_title('AAPL close price (adjusted)', fontsize=13)
axes[0].set_ylabel('Price ($)')

axes[1].bar(df.index, df['volume'], width=1, alpha=0.5, color='steelblue')
axes[1].set_title('Daily volume', fontsize=13)

axes[2].plot(df.index, df['hvol_21d'], linewidth=0.8, color='orangered', label='21d HVol')
axes[2].plot(df.index, df['hvol_63d'], linewidth=0.8, color='steelblue', label='63d HVol', alpha=0.7)
axes[2].set_title('Realised volatility (annualised)', fontsize=13)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- RSI ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

recent = df.last('3Y')
axes[0].plot(recent.index, recent['close'])
axes[0].plot(recent.index, recent['sma_50'],  label='SMA 50',  linewidth=0.8)
axes[0].plot(recent.index, recent['sma_200'], label='SMA 200', linewidth=0.8)
axes[0].set_title('Close + moving averages (last 3 years)', fontsize=13)
axes[0].legend()

axes[1].plot(recent.index, recent['rsi_14'], color='purple', linewidth=0.8)
axes[1].axhline(70, color='red',   linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].set_title('RSI (14)', fontsize=13)
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# --- Label distributions ---
horizons = ['1w', '1m', '3m', '6m', '1y']
fig, axes = plt.subplots(1, len(horizons), figsize=(16, 4))

for ax, h in zip(axes, horizons):
    col = f'dir_{h}'
    counts = df[col].value_counts().sort_index()
    colors = ['#e74c3c', '#95a5a6', '#2ecc71']
    ax.bar(['-1', '0', '+1'], [counts.get(-1,0), counts.get(0,0), counts.get(1,0)], color=colors)
    ax.set_title(f'Direction labels ({h})', fontsize=11)
    ax.set_xlabel('Label')

plt.suptitle('Label distributions across horizons', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature correlation heatmap (subset) ---
feature_cols = ['rsi_14', 'macd_hist', 'bb_pct', 'atr_pct', 'volume_zscore',
                'close_vs_sma50', 'close_vs_sma200', 'hvol_21d', 'stoch_k',
                'ret_1m', 'ret_3m']
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(feature_cols, fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title('Feature correlation matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Summary stats ---
print('=== Dataset summary ===')
print(f'Total trading days : {len(df):,}')
print(f'Features           : {len([c for c in df.columns if not c.startswith(("ret_","dir_","bin_","adj_"))]):,}')
print(f'Label columns      : {len([c for c in df.columns if c.startswith(("ret_","dir_","bin_","adj_"))]):,}')
print()
print('Forward return stats (1-month):')
print(df['ret_1m'].describe().round(4))